# 11 端到端训练示例

## 概述

本笔记本演示大语言模型的**端到端训练流程**——从数据准备到模型生成。

我们将使用仓库中的 LLaMA3 组件，以小配置在随机数据上完成一次完整的训练循环，
涵盖数据准备、模型构建、训练循环、损失可视化、以及推理生成等关键步骤。

### 🧠 直觉理解：训练 LLM 就像教学生写作文

训练一个大语言模型，可以类比为教一个学生写作文：

1. **先学字词（Embedding）**：学生先认识每个字——模型通过 Token Embedding 将离散的 token 映射到连续向量空间
2. **再学语法（Attention）**：学生学会字与字之间的关联——模型通过 Self-Attention 学习 token 间的关系
3. **最后多练习（Training Loop）**：学生反复练习、老师不断纠正——模型通过反向传播和梯度下降不断优化参数

每做一次练习，学生的作文水平就提高一点；每跑一个 step，模型的 loss 就下降一点。

### 📐 数学原理：交叉熵损失与学习率调度

**交叉熵损失（Cross-Entropy Loss）** 是语言模型训练的核心目标函数：

$$\mathcal{L} = -\frac{1}{N} \sum_{i=1}^{N} \log P(y_i \mid x_{<i}; \theta)$$

其中 $y_i$ 是第 $i$ 个目标 token，$x_{<i}$ 是其上文，$\theta$ 是模型参数。
直觉上，交叉熵衡量模型预测分布与真实分布之间的差距——预测越准确，损失越低。

**余弦退火学习率调度（Cosine Annealing）**：

$$\eta_t = \eta_{\min} + \frac{1}{2}(\eta_{\max} - \eta_{\min}) \left(1 + \cos\left(\frac{t}{T_{\max}} \pi\right)\right)$$

学习率先从 $\eta_{\max}$ 平滑下降到 $\eta_{\min}$，帮助模型在训练初期快速收敛，后期精细调优。

In [ ]:
import torch
import torch.nn as nn
import sys; sys.path.insert(0, '..')
from models.llama3.model import LLaMA3ForCausalLM
from models.llama3.config import LLaMA3Config
import matplotlib.pyplot as plt
import numpy as np
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

print(f"PyTorch 版本: {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")

## 1. 数据准备

在真实场景中，数据准备包括：文本收集 → 清洗 → 分词（Tokenization）→ 构造训练样本。

这里我们使用随机整数模拟 token id，演示数据的基本形状和格式。

In [ ]:
# 创建随机训练数据
vocab_size = 256
seq_len = 64
batch_size = 4
n_batches = 20

data = torch.randint(0, vocab_size, (n_batches, batch_size, seq_len))
print(f"训练数据形状: {data.shape}")
print(f"  - 批次数: {n_batches}")
print(f"  - 每批样本数: {batch_size}")
print(f"  - 序列长度: {seq_len}")
print(f"  - 词汇表大小: {vocab_size}")
print(f"\n数据示例 (第一个样本前 16 个 token):")
print(data[0][0][:16].tolist())

## 2. 模型构建

使用仓库中的 LLaMA3 组件，配置一个小型模型用于演示。

关键配置说明：
- `dim=128`：模型隐藏维度（真实 7B 模型为 4096）
- `n_layers=2`：Transformer 层数（真实 7B 模型为 32）
- `n_heads=4, n_kv_heads=2`：GQA 配置（2 组 KV 头共享）

In [ ]:
config = LLaMA3Config(
    vocab_size=vocab_size,
    dim=128,
    n_layers=2,
    n_heads=4,
    n_kv_heads=2,
    ff_hidden_dim=256,
    max_seq_len=seq_len,
)
model = LLaMA3ForCausalLM(config)
n_params = sum(p.numel() for p in model.parameters())
print(f"模型参数量: {n_params:,}")
print(f"\n模型结构概览:")
print(f"  - 嵌入层: {config.vocab_size} × {config.dim} = {config.vocab_size * config.dim:,} 参数")
print(f"  - Transformer 层数: {config.n_layers}")
print(f"  - 注意力头: {config.n_heads} Q 头, {config.n_kv_heads} KV 头")
print(f"  - FFN 中间维度: {config.ff_hidden_dim}")

## 3. 训练循环

训练循环是 LLM 训练的核心，包含以下步骤：

1. **输入构造**：将序列分为输入（前 N-1 个 token）和目标（后 N-1 个 token），即 next-token prediction
2. **前向传播**：模型输出每个位置的 logits
3. **损失计算**：用交叉熵比较预测 logits 与目标 token
4. **反向传播**：计算梯度
5. **参数更新**：优化器根据梯度更新参数
6. **学习率调度**：调整学习率

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_batches)

losses = []
for i in range(n_batches):
    input_ids = data[i][:, :-1]
    targets = data[i][:, 1:]
    
    logits = model(input_ids)
    loss = nn.functional.cross_entropy(
        logits.reshape(-1, vocab_size), targets.reshape(-1)
    )
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    scheduler.step()
    
    losses.append(loss.item())
    if (i + 1) % 5 == 0:
        lr = optimizer.param_groups[0]['lr']
        print(f"Step {i+1}/{n_batches}, Loss: {loss.item():.4f}, LR: {lr:.6f}")

print(f"\n训练完成! 最终 Loss: {losses[-1]:.4f}")

## 4. 训练可视化

观察 Loss 曲线的变化趋势，理想情况下 Loss 应随训练步数逐渐下降。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss 曲线
axes[0].plot(losses, 'b-', linewidth=1.5)
axes[0].set_xlabel('训练步数')
axes[0].set_ylabel('损失值')
axes[0].set_title('训练 Loss 曲线')
axes[0].grid(True, alpha=0.3)

# 学习率曲线
lrs = []
temp_optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
temp_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(temp_optimizer, T_max=n_batches)
for _ in range(n_batches):
    lrs.append(temp_optimizer.param_groups[0]['lr'])
    temp_scheduler.step()

axes[1].plot(lrs, 'r-', linewidth=1.5)
axes[1].set_xlabel('训练步数')
axes[1].set_ylabel('学习率')
axes[1].set_title('余弦退火学习率调度')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. 生成推理

训练完成后，我们可以使用模型进行自回归文本生成。

生成过程：
1. 给定一个 prompt（起始 token 序列）
2. 模型预测下一个 token 的概率分布
3. 根据温度参数采样得到下一个 token
4. 重复直到达到最大生成长度

由于我们使用的是随机数据训练的小模型，生成结果没有语义意义，但可以验证训练流程的正确性。

In [ ]:
prompt = torch.randint(0, vocab_size, (1, 8))
with torch.no_grad():
    generated = model.generate(prompt, max_new_tokens=32, temperature=0.8)
print(f"Prompt:    {prompt.tolist()}")
print(f"Generated: {generated.tolist()}")
print(f"\n生成长度: {generated.shape[1]} (prompt {prompt.shape[1]} + 新生成 {generated.shape[1] - prompt.shape[1]})")

## 6. 训练流程总结

端到端训练的关键要素：

| 步骤 | 关键操作 | 注意事项 |
|------|----------|----------|
| 数据准备 | Tokenization + 构造样本 | 序列长度、batch size 影响显存 |
| 模型构建 | 选择架构和配置 | 参数量决定计算和显存需求 |
| 训练循环 | Forward → Loss → Backward → Update | 梯度裁剪、混合精度 |
| 学习率调度 | Warmup + Cosine Decay | 避免初期学习率过大 |
| 推理生成 | 自回归采样 | 温度、top-k、top-p 控制多样性 |

在真实的大规模训练中，还需要考虑：
- **数据并行 / 张量并行 / 流水线并行**：将训练扩展到多 GPU / 多节点
- **混合精度训练**：使用 FP16/BF16 减少显存占用
- **梯度检查点**：用计算换显存
- **分布式优化器**：ZeRO 等技术分片优化器状态

In [ ]:
# 对比不同温度下的生成效果
prompt = torch.randint(0, vocab_size, (1, 8))
temperatures = [0.1, 0.5, 1.0, 2.0]

print(f"Prompt: {prompt.tolist()[0]}\n")
for temp in temperatures:
    with torch.no_grad():
        gen = model.generate(prompt, max_new_tokens=16, temperature=temp)
    print(f"Temperature={temp:.1f}: {gen.tolist()[0]}")

## 📝 练习题

1. **超参数影响**：将 `n_layers` 从 2 增加到 4，观察参数量和训练 loss 的变化。参数量增长了多少倍？Loss 是否有明显改善？

2. **学习率敏感性**：尝试将学习率从 `1e-3` 改为 `1e-1` 和 `1e-5`，分别观察训练 loss 曲线的变化。学习率过大或过小分别会导致什么问题？

3. **序列长度影响**：将 `seq_len` 从 64 增加到 128 和 256，观察显存占用和训练速度的变化。为什么更长的序列需要更多显存？